# cell-akamai · deterministic offline replay

This notebook uses **SYNTHETIC_FIXTURE** observations deliberately. It requires no Akamai access and proves only calculation/replay properties. It is not evidence of real GPU inference.

In [1]:
from pathlib import Path
import sys, json
HERE=Path.cwd().resolve()
ROOT=HERE if (HERE/"cell_akamai").is_dir() else HERE.parent
sys.path.insert(0,str(ROOT))
from cell_akamai.kinetics import enrich_observations, verify_kinetics
from cell_akamai.custody import make_fco
from cell_akamai.prometheus import parse_metrics

In [2]:
fixture=[
    {"started_mono_ns":1_000_000_000,"ended_mono_ns":3_000_000_000,"completion_tokens":20,"first_content_mono_ns":1_500_000_000,"source_class":"SYNTHETIC_FIXTURE"},
    {"started_mono_ns":4_000_000_000,"ended_mono_ns":6_000_000_000,"completion_tokens":40,"first_content_mono_ns":4_250_000_000,"source_class":"SYNTHETIC_FIXTURE"},
]
rows=enrich_observations(fixture)
assert rows[0]["tokens_per_s"]==10
assert rows[1]["tokens_per_s"]==20
assert abs(rows[1]["tokens_per_s2_inter_request"]-10/3)<1e-12
assert verify_kinetics(rows)["status"]=="PASS"
print("SYNTHETIC_FIXTURE replay PASS")

SYNTHETIC_FIXTURE replay PASS


In [3]:
altered=[dict(row) for row in rows]
altered[1]["tokens_per_s"]=999
assert verify_kinetics(altered)["status"]=="FAILED"
a=make_fco("SyntheticFixture",{"value":1},{"occurrence":1})
b=make_fco("SyntheticFixture",{"value":1},{"occurrence":2})
assert a["content_id"]==b["content_id"] and a["occurrence_id"]!=b["occurrence_id"]
print("Tamper check PASS; content vs occurrence identity PASS")

Tamper check PASS; content vs occurrence identity PASS


In [4]:
sample=parse_metrics('vllm:kv_cache_usage_perc{model_name="fake"} 0.5\n')
assert sample[0]["value"]==0.5
print("Prometheus units PASS; signed: NO; MMR committed: NO")

Prometheus units PASS; signed: NO; MMR committed: NO
